# Metastasis data analysis

This notebook recreates the results fo the table 3 of the paper: "Reconstructing unobserved cellular states from  paired single-cell lineage tracing and transcriptomics data" paper.

# 0. Standard imports

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd

*** Go to top level directory ***

In [2]:
cd ..

/home/anaf/Study/HTIN5005/A1/treeVAE-reproducibility/scvi/external


In [3]:
cd ..

/home/anaf/Study/HTIN5005/A1/treeVAE-reproducibility/scvi


*** Imports ***

In [7]:
# Tree
from ete3 import Tree
import ete3

# Data
from anndata import AnnData
import matplotlib.pyplot as plt
from external.dataset.tree import TreeDataset, GeneExpressionDataset
from external.dataset.anndataset import AnnDatasetFromAnnData

# Models
from external.models.vae import VAE
import scanpy as sc
from external.inference.tree_inference import TreeTrainer
from external.inference.inference import UnsupervisedTrainer
from external.inference import posterior
from external.models.treevae import TreeVAE

# Utils
from external.utils.data_util import get_leaves, get_internal
from external.utils.plots_util import plot_histograms, plot_density
from external.utils.plots_util import plot_one_gene, training_dashboard
from external.utils.baselines import scvi_baseline_z, cascvi_baseline_z, avg_baseline_z, construct_latent
from external.utils.metrics import knn_purity_tree

# 1. Data Loading

### Import tree

In contrary to the tree used in simulations, this tree is not binary (and therefore requires using the message passing algorithm generalization for multifurcating trees described in the appendix)

The leaves of this tree corresponds to a single clone of 603 cells from a recent dataset that traced the lineages of lung cancer tumors as they metastasized throughout a mouse

In [8]:
tree_name = "data/metastasis/lg7_tree_hybrid_priors.alleleThresh.processed.ultrametric.annotated.tree"

tree = Tree(tree_name, 1)

N = len([n for n in tree.traverse()])
leaves = [n for n in tree.traverse('levelorder') if n.is_leaf()]
print("The tree contains {} nodes and {} leaves".format(N, len(leaves)))

The tree contains 916 nodes and 603 leaves


*Tree preprocessing*: 

As it has (empirically) shown better stability during training, we scale the branch lengths of the tree with the diameter of the tree (which is equal to the maximal distance between the root and its leaves). 

In [9]:
radius = tree.get_leaves()[0].get_distance(tree)
diameter = 2 * radius
print("The diameter of the tree is {}".format(diameter))


branch_length = {}
for node in tree.traverse('levelorder'):
    if node.is_root():
        branch_length[node.name] = 0.0
    else:
        branch_length[node.name] = node.dist / radius
branch_length['prior_root'] = 1.0

The diameter of the tree is 28.0


### Gene expression data

A large fraction of genes detected by scRNA-seq do not have a strong relationship to the lineage, we only considered the top 100 genes autocorrelated with the phylogeny, as evaluated by *Hotspot* (https://github.com/YosefLab/Hotspot) 

We import the preprocessed gene expresion matrix

In [10]:
X = np.load('data/metastasis/Metastasis_lg7_100g.npy')
print("the matrix contains {} cells and {} genes".format(X.shape[0], X.shape[1]))

the matrix contains 603 cells and 100 genes


# 2. Fitting treeVAE

In [11]:
import torch
    
np.random.seed(42)
torch.manual_seed(42)

***Anndata***

In [12]:
# Anndata + gene and cells filtering
adata = AnnData(X)
adata.obs_names = [n.name for n in leaves]
sc.pp.filter_genes(adata, min_counts=3)
sc.pp.filter_cells(adata, min_counts=0)

***Create a TreeDataset object***

In [13]:
# treeVAE
import copy

n_cells, n_genes = X.shape
tree_bis = copy.deepcopy(tree)
scvi_dataset = AnnDatasetFromAnnData(adata, filtering=False)
scvi_dataset.initialize_cell_attribute('barcodes', adata.obs_names)
cas_dataset = TreeDataset(scvi_dataset, tree=tree, filtering=False)
cas_dataset

# No batches beacause of the message passing
use_cuda = True
use_MP = True
ldvae = False

n_counts is a protected attribute or already exists as a cell attribute and cannot be set with this name in initialize_gene_attribute, changing name to n_counts_gene and setting


go


***Initialize model***

In [14]:
treevae = TreeVAE(cas_dataset.nb_genes,
              tree = cas_dataset.tree,
              n_latent=10,
              n_hidden=64,
              n_layers=1,
              reconstruction_loss='poisson',
              prior_t = branch_length,
              ldvae = ldvae,
              use_MP=use_MP
             )

***Training hyperparameters***

In [15]:
n_epochs = 700
lr = 1e-3
lambda_ = 1.0

***Trainer***

In [16]:
freq = 100
trainer = TreeTrainer(
    model = treevae,
    gene_dataset = cas_dataset,
    lambda_ = lambda_,
    train_size=1.0,
    test_size=0,
    use_cuda=use_cuda,
    frequency=freq,
    n_epochs_kl_warmup=150
)

train_leaves:  [[np.int64(0)], [np.int64(1)], [np.int64(2)], [np.int64(3)], [np.int64(4)], [np.int64(5)], [np.int64(6)], [np.int64(7)], [np.int64(8)], [np.int64(9)], [np.int64(10)], [np.int64(11)], [np.int64(12)], [np.int64(13)], [np.int64(14)], [np.int64(15)], [np.int64(16)], [np.int64(17)], [np.int64(18)], [np.int64(19)], [np.int64(20)], [np.int64(21)], [np.int64(22)], [np.int64(23)], [np.int64(24)], [np.int64(25)], [np.int64(26)], [np.int64(27)], [np.int64(28)], [np.int64(29)], [np.int64(30)], [np.int64(31)], [np.int64(32)], [np.int64(33)], [np.int64(34)], [np.int64(35)], [np.int64(36)], [np.int64(37)], [np.int64(38)], [np.int64(39)], [np.int64(40)], [np.int64(41)], [np.int64(42)], [np.int64(43)], [np.int64(44)], [np.int64(45)], [np.int64(46)], [np.int64(47)], [np.int64(48)], [np.int64(49)], [np.int64(50)], [np.int64(51)], [np.int64(52)], [np.int64(53)], [np.int64(54)], [np.int64(55)], [np.int64(56)], [np.int64(57)], [np.int64(58)], [np.int64(59)], [np.int64(60)], [np.int64(61)], [n

***Start training***

In [17]:
trainer.train(n_epochs=n_epochs,
              lr=lr
              )

training:   0%|          | 0/700 [00:00<?, ?it/s]Encodings MP Likelihood: 43.72291441023013
ELBO Loss: 2713.241225126829
training:   0%|          | 1/700 [00:00<03:28,  3.35it/s]Encodings MP Likelihood: 43.61446481291556
ELBO Loss: 2621.631101256023
training:   0%|          | 2/700 [00:00<02:32,  4.57it/s]Encodings MP Likelihood: 44.656530939969514
ELBO Loss: 2543.3667398277607
training:   0%|          | 3/700 [00:00<02:12,  5.27it/s]Encodings MP Likelihood: 44.91604630887808
ELBO Loss: 2473.026886807354
training:   1%|          | 4/700 [00:00<02:05,  5.56it/s]Encodings MP Likelihood: 46.18425800624305
ELBO Loss: 2404.7692186541476
training:   1%|          | 5/700 [00:00<02:00,  5.78it/s]Encodings MP Likelihood: 43.799542001235984
ELBO Loss: 2334.4510462356416
training:   1%|          | 6/700 [00:01<01:55,  5.99it/s]Encodings MP Likelihood: 45.82115658946476
ELBO Loss: 2260.7403485650534
training:   1%|          | 7/700 [00:01<01:52,  6.15it/s]Encodings MP Likelihood: 44.41419336993306

***Training dashboard***

the learning curves will be saved in a temporary file '*training_cascvi.png*'

In [18]:
training_dashboard(trainer, treevae.encoder_variance)

***Training ELBO***

In [19]:
full_posterior = trainer.create_posterior(trainer.model, cas_dataset, trainer.clades,
                                indices=np.arange(len(cas_dataset))
                                         )

print("treeVAE training ELBO: {}".format(full_posterior.compute_elbo().item()))

treeVAE training ELBO: 263.0483775084111


# 3. Fitting scVI

In [20]:
scVI = TreeVAE(cas_dataset.nb_genes,
              tree = cas_dataset.tree,
              n_latent=10,
              n_hidden=64,
              n_layers=1,
              reconstruction_loss='poisson',
              prior_t = branch_length,
              ldvae = ldvae,
              use_MP=False
             )

freq = 100
trainer_scvi = TreeTrainer(
    model = scVI,
    gene_dataset = cas_dataset,
    lambda_ = lambda_,
    train_size=1.0,
    test_size=0,
    use_cuda=use_cuda,
    frequency=freq,
    n_epochs_kl_warmup=150
)

trainer_scvi.train(n_epochs=n_epochs,
              lr=lr
              )

train_leaves:  [[np.int64(0)], [np.int64(1)], [np.int64(2)], [np.int64(3)], [np.int64(4)], [np.int64(5)], [np.int64(6)], [np.int64(7)], [np.int64(8)], [np.int64(9)], [np.int64(10)], [np.int64(11)], [np.int64(12)], [np.int64(13)], [np.int64(14)], [np.int64(15)], [np.int64(16)], [np.int64(17)], [np.int64(18)], [np.int64(19)], [np.int64(20)], [np.int64(21)], [np.int64(22)], [np.int64(23)], [np.int64(24)], [np.int64(25)], [np.int64(26)], [np.int64(27)], [np.int64(28)], [np.int64(29)], [np.int64(30)], [np.int64(31)], [np.int64(32)], [np.int64(33)], [np.int64(34)], [np.int64(35)], [np.int64(36)], [np.int64(37)], [np.int64(38)], [np.int64(39)], [np.int64(40)], [np.int64(41)], [np.int64(42)], [np.int64(43)], [np.int64(44)], [np.int64(45)], [np.int64(46)], [np.int64(47)], [np.int64(48)], [np.int64(49)], [np.int64(50)], [np.int64(51)], [np.int64(52)], [np.int64(53)], [np.int64(54)], [np.int64(55)], [np.int64(56)], [np.int64(57)], [np.int64(58)], [np.int64(59)], [np.int64(60)], [np.int64(61)], [n

In [22]:
scvi_posterior = trainer_scvi.create_posterior(trainer_scvi.model, cas_dataset, trainer_scvi.clades,
                                indices=np.arange(len(cas_dataset))
                                         )

print("scVI - full - batch train ELBO: {}".format(scvi_posterior.compute_elbo().item()))

scVI - full - batch train ELBO: 263.58464562622254


# 5. Average ELBO

We compute the average ELBO on 100 forward passes on the training set.

In [23]:
elbo_treevae  = []
elbo_vae = []
N = 100
for i in range(N):
    elbo_vae.append(scvi_posterior.compute_elbo().item())
    elbo_treevae.append(full_posterior.compute_elbo().item())

treeVAE averaged ELBO:

In [24]:
print("treeVAE Elbo: mean = {} | std = {}".format(np.mean(elbo_treevae), np.std(elbo_treevae)))

treeVAE Elbo: mean = 263.05525793031114 | std = 0.4015809911742225


scVI averaged ELBO:

In [25]:
print("scVI Elbo: mean = {} | std = {}".format(np.mean(elbo_vae), np.std(elbo_vae)))

scVI Elbo: mean = 263.5319928989161 | std = 0.36412016360226157


# 6. Cross-Entropy

In [26]:
treevae_latent = full_posterior.get_latent()
scvi_latent = scvi_posterior.get_latent()

We compute the cross-entropy for scVI's latent space

In [27]:
treevae.initialize_visit()
treevae.initialize_messages(scvi_latent, cas_dataset.barcodes, scvi_latent.shape[1])
treevae.perform_message_passing((treevae.tree & treevae.root), scvi_latent.shape[1], False)
mp_lik_scvi = treevae.aggregate_messages_into_leaves_likelihood(10, add_prior=True)
print("scVI Cross-Entropy: ", mp_lik_scvi.item())

scVI Cross-Entropy:  -16387.225210635235


We compute the cross-entropy for treeVAE's latent space

In [28]:
treevae.initialize_visit()
treevae.initialize_messages(treevae_latent, cas_dataset.barcodes, treevae_latent.shape[1])
treevae.perform_message_passing((treevae.tree & treevae.root), treevae_latent.shape[1], False)
mp_lik_cascvi = treevae.aggregate_messages_into_leaves_likelihood(10, add_prior=True)
print("Likelihood of cascVI encodings: ", mp_lik_cascvi.item())

Likelihood of cascVI encodings:  -4263.225933368372
